# Two Machines, One Environment

In this short lab, one neural network learns an economic policy function and one miniature transformer generates central-bank prose. Both run offline on CPU from bundled assets.

**Live path:** checkout -> growth policy -> MiniGPT -> change one dial. No API key, network connection, GPU, installation, or public presentation is required.

## Pre-class setup only

Package installation is preparation, not part of the fifteen-minute live path. Deck 1.5 walks through it: clone the course repository, create its environment (`python -m venv .venv`), activate it, then install, from the repository root:

```bash
python -m pip install -i https://pypi.tuna.tsinghua.edu.cn/simple numpy pandas matplotlib scipy scikit-learn torch jupyter ipykernel
```

These eight packages cover every Session lab. In VS Code, select the `.venv` kernel for this notebook. If the checkout below reports a missing package or asset, fix that first; do not fetch a model during class.

## 1. System checkout

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import shutil
import sys

# 1. The workbench: is every package the six labs use installed in *this* kernel's Python?
NEEDED = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'scipy': 'scipy',
          'sklearn': 'scikit-learn', 'torch': 'torch', 'ipykernel': 'ipykernel'}
print(f"{'python':14s}: {platform.python_version()}  ({sys.executable})")
missing = []
for module, package in NEEDED.items():
    found = importlib.util.find_spec(module) is not None
    print(f"{package:14s}: {importlib.metadata.version(package) if found else 'MISSING'}")
    if not found:
        missing.append(package)
git_found = shutil.which('git') is not None
print(f"{'git':14s}: {'found' if git_found else 'MISSING'}")
if missing or not git_found:
    raise SystemExit('NOT READY: install ' + ' '.join(missing + ([] if git_found else ['git']))
                     + " into this kernel's environment (see the setup cell above).")

# 2. The course folder: the lab reads helpers and data from the folders beside it.
here = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in (here, *here.parents)
                  if (candidate / 'labs' / 'session01_assets' / 'session01_demo.py').is_file()), None)
if REPO_ROOT is None:
    raise SystemExit('NOT READY: open this notebook from inside the cloned course folder; '
                     'it reads helpers and data from the folders beside it.')
OUTPUT_DIR = REPO_ROOT / 'labs' / 'outputs' / 'session01'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MPL_CACHE = OUTPUT_DIR / 'matplotlib'
MPL_CACHE.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE))
sys.path.insert(0, str(REPO_ROOT / 'labs' / 'session01_assets'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import session01_demo as demo

checkout = demo.environment_report()
for key, value in checkout.items():
    print(f'{key:18s}: {value}')
assert checkout['ready'], 'A bundled Session 1 asset is missing; refresh the repository.'
print('\nREADY: offline CPU demos can run.')

In [ ]:
# Verify the bundled FOMC corpus against the checkpoint training report.
asset_dir = REPO_ROOT / 'labs' / 'session01_assets'
training_report = json.loads((asset_dir / 'training_report.json').read_text(encoding='utf-8'))
fomc_dir = REPO_ROOT / 'labs' / 'Lec07_LLM_Lab' / 'data' / 'fomc'
statements = []
for path in sorted(fomc_dir.glob('fomc_*.txt')):
    lines = path.read_text(encoding='utf-8').splitlines()
    body = '\n'.join(line for line in lines if not line.startswith('#')).strip()
    if body:
        statements.append(body)
corpus = '\n\n'.join(statements)
corpus_hash = hashlib.sha256(corpus.encode('utf-8')).hexdigest()
assert len(statements) == training_report['n_statements']
assert len(corpus) == training_report['corpus_characters']
assert corpus_hash == training_report['corpus_sha256']
print(f"FOMC corpus verified: {len(statements)} statements, {len(corpus):,} characters")
print('SHA-256:', corpus_hash)

## 2. Machine A: learn an optimal-growth policy

The household chooses tomorrow's capital in a full-depreciation growth model:

$$\max \sum_{t=0}^{\infty} \beta^t \log c_t, \qquad c_t + k_{t+1}=A k_t^\alpha.$$

The network is feasible by construction: $k'=Ak^\alpha\,\sigma(g_\theta(\log k))$. Training minimizes the squared log Euler-equation residual. The analytical policy $k'=\alpha\beta Ak^\alpha$ is hidden from training and used only for validation.

In [ ]:
torch.manual_seed(2026)
random_policy = demo.GrowthPolicy()
before = demo.growth_diagnostics(random_policy)

learned_policy, loss_history, elapsed = demo.train_growth_policy(
    steps=500, batch_size=256, seed=2026, learning_rate=3e-3
)
after = demo.growth_diagnostics(learned_policy)
used_reference = False

# The normal path is always from-scratch training. The checkpoint is a classroom fallback.
if (not np.isfinite(after['median_relative_error'])
        or after['median_relative_error'] >= 0.02):
    learned_policy, reference_report = demo.load_growth_reference()
    after = demo.growth_diagnostics(learned_policy)
    used_reference = True

grid = torch.tensor(after['capital'], dtype=torch.float32).reshape(-1, 1)
with torch.no_grad():
    choices = learned_policy(grid)
    resources = learned_policy.resources(grid)
assert torch.all(choices > 0) and torch.all(choices < resources)
assert after['median_relative_error'] < 0.02

print(f'from-scratch runtime: {elapsed:.3f} seconds')
print(f'reference fallback used: {used_reference}')
print(f"median relative policy error: {100 * after['median_relative_error']:.3f}%")
print(f"out-of-batch Euler residual, p95: {after['residual_p95']:.6f}")
print('feasibility: PASS (0 < k_next < output on every validation state)')

In [ ]:
fig = demo.plot_growth_before_after(before, after, loss_history)
fig.savefig(OUTPUT_DIR / 'growth_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Machine B: generate with a MiniGPT

This is the decoder-only transformer architecture from Lab 7B, loaded from a state-dict-only checkpoint trained on the bundled FOMC corpus. It predicts characters, not facts.

Choose one prompt from `PROMPTS` and one temperature from `TEMPERATURES`. Lower temperature concentrates probability on familiar continuations; higher temperature produces more variation and more errors.

In [ ]:
mini_gpt, stoi, vocabulary, report = demo.load_minigpt()
parameter_count = sum(parameter.numel() for parameter in mini_gpt.parameters())
print(f'MiniGPT loaded: {parameter_count:,} parameters')
print(f"validation loss at checkpoint: {report['final_validation_loss']:.4f}")
print('checkpoint SHA-256:', report['checkpoint_sha256'])

In [ ]:
PROMPTS = ('The Committee decided', 'Inflation remains', 'Economic activity')
TEMPERATURES = (0.5, 0.8, 1.2)

prompt = PROMPTS[0]          # bounded choice: 0, 1, or 2
temperature = TEMPERATURES[1]  # bounded choice: 0, 1, or 2

synthetic = demo.generate_text(
    mini_gpt, stoi, vocabulary, prompt=prompt, n_new=200,
    temperature=temperature, seed=2026
)
print('SYNTHETIC MINI-GPT OUTPUT - POTENTIALLY FALSE')
print(f'prompt={prompt!r}, temperature={temperature}')
print('-' * 72)
print(synthetic)
print('-' * 72)

In [ ]:
# Acceptance checks: fixed seeds reproduce; temperature changes the sampled path.
repeat = demo.generate_text(
    mini_gpt, stoi, vocabulary, prompt=prompt, n_new=200,
    temperature=temperature, seed=2026
)
low = demo.generate_text(
    mini_gpt, stoi, vocabulary, prompt=prompt, n_new=120,
    temperature=0.5, seed=2027
)
high = demo.generate_text(
    mini_gpt, stoi, vocabulary, prompt=prompt, n_new=120,
    temperature=1.2, seed=2027
)
assert repeat == synthetic
assert low != high
print('fixed-seed reproduction: PASS')
print('temperature sensitivity: PASS')

### Change one dial

Change only `prompt` or `temperature` in the generation cell, then privately note:

- Did fluency change?
- Did repetition change?
- Did factual reliability improve, or did the prose only sound different?

There is no group presentation or cold call.

## 4. What the two machines preview

| What you saw | Later owner |
|---|---|
| Neural policy and Euler residual | Session 2 |
| Sequential decisions and value | Session 3 |
| Distributions and equilibrium | Session 4 |
| Tokens, attention, and GPT training | Session 5 |
| Fluent but unsupported output | Sessions 5-6 |
| Grounding and research agents | Session 6 |

The shared lesson is simple: approximation creates an output; validation turns it into evidence.